# AgentsVille Trip Planner - Project Assignment

In this project, you'll implement an AI system to help you plan a trip--to the wonderful city of AgentsVille!

Your AI system will demonstrate advanced LLM reasoning techniques including:

1. **Role-Based Prompting** - Your agent will act as a specialized travel planner
2. **Chain-of-Thought Reasoning** - Step-by-step planning of itineraries
3. **ReAct Prompting** - Thought → Action → Observation cycles
4. **Feedback Loops** - Self-evaluation using tools in the ReAct loop to find mistakes and improve plans

You'll simulate external API calls to gather weather data and activities. Then, process this information to create personalized travel itineraries based on interests and other constraints. Last, you'll implement a feedback loop to refine the plan.

Your task is to build a travel agent that can plan the perfect AgentsVille vacation!

## Project Instructions

Here are the steps you'll follow:

1. **Define Vacation Details**
    - Specify the trip duration, interests, and constraints.
    - Use Pydantic to structure and verify this information in a class called `VacationInfo`.
2. **Review Weather and Activity schedules**
    - Simulate API calls to gather weather data and available activities in bulk
    - Review the data manually to understand the available options
3. **The ItineraryAgent**
    - Implement an agent that generates a day-by-day itinerary based on the vacation details
    - The system prompt will guide the agent's reasoning through a step-by-step planning process to take travel preferences (e.g. destination, dates, interests) and generate a detailed day-by-day itinerary
    - Craft the components of the prompt (including the role, task/instructions, output format, examples, and context) to elicit the best possible itinerary in one LLM call
4. **Evaluating the Itinerary**
    - Evaluate the itinerary using a set of criteria to ensure a high-quality travel plan
        - For instance, does the itinerary match the city and the dates requested?
        - Or, is the total cost calulation accurate and is it within budget?
        - Or, does the agent hallucinate any activities that are not available?
        - Or, does the agent suggest activities that are not suitable for the weather? This specific evaluation function will require the use of an LLM to compare the event description against the weather data.
5. **Defining the Tools**
    - We will define four tools to assist the agent
        - **calculator_tool**: to accurately calculate costs
        - **get_activities_by_date_tool**: to retrieve activities for a specific date
        - **run_evals_tool**: to evaluate the itinerary against the criteria
        - **final_answer_tool**: to provide the final answer in a structured format
6. **The ItineraryRevisionAgent**
    - We will implement a second agent that revises the itinerary based on feedback using the ReAct THOUGHT → ACTION → OBSERVATION cycle.
        - The LLM will first generated a THOUGHT / ACTION message, which contains reasoning steps and a tool call invocation.
        - The Python code will parse the tool call and execute it, returning the result as a string to the LLM in an OBSERVATION message.
        - After this cycle repeats n number of times, the LLM will invoke the final_answer_tool to signal to the Python code to end the loop and return the final answer.
    - This agent will also **incorporate feedback on the initial itinerary** from the travelers to ensure the final plan has **at least 2 activities per day**. A new evaluation function using a powerful LLM will be created to check this user feedback.
    - The agent will use the tools above to refine the plan iteratively, checking the weather and available activities, and ensuring the itinerary meets all constraints.
7. **Something just for fun!**
    - To wrap things up we'll create a fun, narrative summary of the trip, highlighting the best activities and experiences!

## Initial Setup

Let's start with settin up our environment and defining the vacation details.

In [1]:
# When using VSCode in the Udacity workspace, add /workspace to the PYTHON_PATH
import os
import sys
from dotenv import load_dotenv

WORKSPACE_DIRECTORY = "/workspace"
if os.path.exists(WORKSPACE_DIRECTORY) and WORKSPACE_DIRECTORY not in sys.path:
    sys.path.append(WORKSPACE_DIRECTORY)
    print(f"Added {WORKSPACE_DIRECTORY} to the Python path")
# Load environment variables from .env
load_dotenv()

True

In [2]:
# Install required packages if not already installed
# No changes needed here.
%pip install -q json-repair==0.47.1 numexpr==2.11.0 openai==1.74.0 pandas==2.3.0 pydantic==2.11.7 python-dotenv==1.1.0

Note: you may need to restart the kernel to use updated packages.


In [3]:
# If using the Vocareum API endpoint
# TODO: Fill in the missing parts marked with **********
from openai import OpenAI

client = OpenAI(
    # Change the base_url when using the Vocareum API endpoint
    # If using the OpenAI API endpoint, you can comment out the base_url line
    base_url="https://openai.vocareum.com/v1",
    # Uncomment one of the following
    api_key=os.getenv("OPENAI_API_KEY_DEV", ""),  # <--- TODO: Fill in your Vocareum API key here
    # api_key=os.getenv(
    #     "OPENAI_API_KEY"
    # ),  # <-- Alternately, set as an environment variable
)


In [4]:
# Throughout this project you can experiment with different OpenAI models.
# By default we will use GPT-4.1-mini, which is a good balance of speed and cost.
from enum import Enum

class OpenAIModel(str, Enum):
    GPT_41 = "gpt-4.1"  # Strong default choice for development tasks, particularly those requiring speed, responsiveness, and general-purpose reasoning. 
    GPT_41_MINI = "gpt-4.1-mini"  # Fast and affordable, good for brainstorming, drafting, and tasks that don't require the full power of GPT-4.1.
    GPT_41_NANO = "gpt-4.1-nano"  # The fastest and cheapest model, suitable for lightweight tasks, high-frequency usage, and edge computing.

MODEL = OpenAIModel.GPT_41_MINI  # Default model for this project


## Define Vacation Details

Let's encode the details of our vacation in JSON format and verify it using Pydantic.

In practice, a chatbot agent could gather the information of a user. After it has gathered all the information it needs, it could generate this JSON object from the chat transcript. We will skip that step to focus on the itinerary generation itself.

In [5]:
# The Vacation Info Data Structure
# No changes needed here, but you may choose to personalize the data.

VACATION_INFO_DICT = {
    "travelers": [
        {
            "name": "Yuri",
            "age": 30,
            # Possible interests: art, cooking, comedy, dancing, fitness, gardening, hiking, math, movies,
            # music, nutrition, photography, programming, reading, soccer, sports, technology, theatre, tennis, writing
            "interests": ["tennis", "cooking", "comedy", "technology"],
        },
        {
            "name": "Hiro",
            "age": 25,
            # Possible interests: art, cooking, comedy, dancing, fitness, gardening, hiking, math, movies,
            # music, nutrition, photography, programming, reading, soccer, sports, technology, theatre, tennis, writing
            "interests": ["reading", "music", "theatre", "art"],
        },
        {
            "name": "Kaoru Kishimoto",
            "age": 54,
            # Possible interests: art, cooking, comedy, dancing, fitness, gardening, hiking, math, movies,
            # music, nutrition, photography, programming, reading, soccer, sports, technology, theatre, tennis, writing
            "interests": ["math", "programming", "soccer", "nutrition"],
        },
    ],
    "destination": "AgentsVille",
    "date_of_arrival": "2026-03-01",  # Mock data exists for 2026-03-01
    "date_of_departure": "2026-03-08",  # ...until 2026-03-08.
    "budget": 300,  # Budget is in fictional currency units.
}

In [6]:
# Validate the data structure using Pydantic
# TODO: Fill in the missing parts marked with **********

from project_lib import Interest
from typing import List
from pydantic import BaseModel
import datetime
from pprint import pprint

class Traveler(BaseModel):
    """A traveler with a name, age, and list of interests.
    
    Attributes:
        name (str): The name of the traveler.
        age (int): The age of the traveler.
        interests (List[Interest]): A list of interests of the traveler.
    """
    name: str
    age: int
    interests: List[Interest]

class VacationInfo(BaseModel):
    """Vacation information including travelers, destination, dates, and budget.
    Attributes:
        travelers (List[Traveler]): A list of travelers.
        destination (str): The vacation destination.
        date_of_arrival (datetime.date): The date of arrival.
        date_of_departure (datetime.date): The date of departure.
        budget (int): The budget for the vacation in fictional currency units.
    """
    # TODO: Fill in the the missing fields for the VacationInfo class
    travelers: List[Traveler]
    destination: str
    date_of_arrival: datetime.date
    date_of_departure: datetime.date
    budget: int


# Validate the VacationInfo data structure
vacation_info = VacationInfo.model_validate(VACATION_INFO_DICT)

# Display the vacation info as a dictionary
pprint(vacation_info.model_dump())

# Check that VacationInfo contains the expected data
assert "travelers" in vacation_info.model_dump().keys(), "VacationInfo should contain 'travelers' key"
assert "destination" in vacation_info.model_dump().keys(), "VacationInfo should contain 'destination' key"
assert "date_of_arrival" in vacation_info.model_dump().keys(), "VacationInfo should contain 'date_of_arrival' key"
assert "date_of_departure" in vacation_info.model_dump().keys(), "VacationInfo should contain 'date_of_departure' key"
assert "budget" in vacation_info.model_dump().keys(), "VacationInfo should contain 'budget' key"
assert isinstance(vacation_info.travelers, list), "Travelers should be a list"
assert all(isinstance(traveler, Traveler) for traveler in vacation_info.travelers), "All travelers should be instances of Traveler class"
assert isinstance(vacation_info.date_of_arrival, datetime.date), "date_of_arrival should be a date"
assert isinstance(vacation_info.date_of_departure, datetime.date), "date_of_departure should be a date"
assert isinstance(vacation_info.budget, int), "budget should be an integer"

# If all assertions pass, print a success message
print("✅ VacationInfo data structure is valid!")

{'budget': 300,
 'date_of_arrival': datetime.date(2026, 3, 1),
 'date_of_departure': datetime.date(2026, 3, 8),
 'destination': 'AgentsVille',
 'travelers': [{'age': 30,
                'interests': [tennis, cooking, comedy, technology],
                'name': 'Yuri'},
               {'age': 25,
                'interests': [reading, music, theatre, art],
                'name': 'Hiro'},
               {'age': 54,
                'interests': [math, programming, soccer, nutrition],
                'name': 'Kaoru Kishimoto'}]}
✅ VacationInfo data structure is valid!


## Review Weather and Activity Schedules

Now that we have the trip details, we can retrieve the weather and activity schedules for the dates of the trip.

We will call an API to get all the data at once, in order to be able to include it in the context for our itinerary planning agent.

Also, we will format the data as Pandas DataFrames for easier viewing. Take the time to read and understand the data to see if the agent notices the same things you do. For instance:
- What does the weather look like for the trip? On what days it is sunny, rainy, or cloudy?
- What activities are available on each day? Are there any special events or festivals related to the user's interests?

In [7]:
# The `call_weather_api_mocked` mocks calling a weather API to get weather data
# TODO: Fill in the missing parts marked with **********

from project_lib import call_weather_api_mocked
import pandas as pd

pd.set_option("display.max_colwidth", None)  # Show full content in DataFrame cells

weather_for_dates = [
    call_weather_api_mocked(
        date=ts.strftime("%Y-%m-%d"), city=vacation_info.destination
    )
    for ts in pd.date_range(
        # TODO: Fill in the missing start and end dates from vacation_info
        start=vacation_info.date_of_arrival,
        end=vacation_info.date_of_departure,
        freq="D",
    )
]

weather_for_dates_df = pd.DataFrame(weather_for_dates)

weather_for_dates_df

,date,city,temperature,temperature_unit,condition,description
0,2026-03-01,AgentsVille,22,celsius,partly cloudy,Mild temperatures with a mix of sun and clouds. Comfortable for outdoor activities.
1,2026-03-02,AgentsVille,24,celsius,sunny,Sunny skies and pleasant warmth throughout the day. Great for parks and walking tours.
2,2026-03-03,AgentsVille,20,celsius,rainy,Overcast with intermittent rain showers and a light breeze. Bring an umbrella.
3,2026-03-04,AgentsVille,18,celsius,cloudy,Cooler day with thick cloud cover and no significant precipitation.
4,2026-03-05,AgentsVille,19,celsius,thunderstorm,Afternoon thunderstorms are likely with heavy rain and gusty winds. Outdoor plans should be limited.
5,2026-03-06,AgentsVille,21,celsius,partly cloudy,Morning clouds giving way to partial sun. Mild and comfortable for most activities.
6,2026-03-07,AgentsVille,23,celsius,clear,Clear skies with pleasant temperatures and low humidity. Ideal for outdoor events.
7,2026-03-08,AgentsVille,17,celsius,rainy,Cool and rainy for much of the day with steady showers. Indoor activities recommended.


In [8]:
# The `call_activities_api_mocked` function returns the activities for a given date and city.
# TODO: Fill in the missing parts marked with **********

from project_lib import call_activities_api_mocked

activities_for_dates = [
    activity
    for ts in pd.date_range(
        # TODO: Fill in the missing start and end dates from vacation_info
        start=vacation_info.date_of_arrival,
        end=vacation_info.date_of_departure,
        freq="D",
    )
    for activity in call_activities_api_mocked(
        date=ts.strftime("%Y-%m-%d"), city=vacation_info.destination
    )
]

activities_for_dates_df = pd.DataFrame(activities_for_dates)

activities_for_dates_df

,activity_id,name,start_time,end_time,location,description,price,related_interests
0,event-2026-03-01-0,Tech & Math Brunch Meetup,2026-03-01 09:00,2026-03-01 11:00,"Innovation Loft Cafe, AgentsVille","Kick off your trip with a cozy indoor brunch focused on technology trends and fun math puzzles. Enjoy light talks, group challenges, and plenty of coffee while meeting fellow tech and math enthusiasts.",20,"[technology, math, programming]"
1,event-2026-03-01-1,Riverfront Soccer Skills Clinic,2026-03-01 12:00,2026-03-01 13:30,"Riverside Fields, AgentsVille","Join a friendly soccer clinic focused on drills, footwork, and teamwork by the river. This outdoor session includes a warm-up and short scrimmage; in case of rain, we move to the nearby Community Sports Hall.",15,"[soccer, fitness]"
2,event-2026-03-01-2,Nutrition Market Tasting Tour,2026-03-01 15:00,2026-03-01 17:00,"Harvest Market District, AgentsVille","Explore AgentsVille's freshest ingredients on a guided market tour. Learn about seasonal produce, balanced meals, and cooking tips with tastings at select stalls. Includes indoor vendor stops for rainy moments.",18,"[nutrition, cooking]"
3,event-2026-03-01-3,Evening Comedy Showcase,2026-03-01 19:00,2026-03-01 20:30,"Laughline Theater, AgentsVille",Relax indoors with a curated lineup of local comedians. A lighthearted night of stand-up and sketches perfect for comedy fans.,15,[comedy]
4,event-2026-03-02-0,Sunrise Photography Walk,2026-03-02 08:00,2026-03-02 10:00,"Old Town Promenade, AgentsVille",Capture golden morning light on a guided photography walk through historic streets and scenic viewpoints. Outdoor event with indoor gallery stop included.,15,"[photography, art]"
5,event-2026-03-02-1,AgentsVille Soccer Friendly,2026-03-02 12:00,2026-03-02 13:30,"Sunfield Park Pitch, AgentsVille","Join a casual outdoor soccer match with locals. All skill levels welcome for a fun, energetic mid-day game.",10,"[soccer, fitness]"
6,event-2026-03-02-2,Open-Air Coding Picnic,2026-03-02 15:00,2026-03-02 17:00,"Central Park Lawn, AgentsVille","Bring a laptop or notebook for a relaxed outdoor programming jam. Share tips, solve mini challenges, and enjoy snacks in the park.",12,"[programming, technology]"
7,event-2026-03-02-3,Future Apps Tech Meetup,2026-03-02 19:00,2026-03-02 20:30,"The Digital Atrium, AgentsVille","An indoor evening meetup featuring lightning talks on app development, product design, and emerging tech trends. Network with fellow builders and creatives.",20,"[technology, programming]"
8,event-2026-03-03-0,Modern Art Museum Highlights,2026-03-03 09:30,2026-03-03 11:30,AgentsVille Modern Art Museum,"Stay dry with a guided indoor tour of contemporary exhibits, artist talks, and interactive installations.",18,"[art, photography]"
9,event-2026-03-03-1,Indoor Tennis Skills Lab,2026-03-03 12:30,2026-03-03 14:00,"Grand Courts Indoor Center, AgentsVille",Sharpen your serve and footwork in a climate-controlled indoor tennis clinic led by local coaches.,16,"[tennis, fitness]"


## The ItineraryAgent

First we will review the Pydantic objects used for defining the output of our agent, the TravelPlan, ItineraryDay, Activity, and Weather classes.

Second, we will create a Chain-of-Thought prompt to guide the agent in planning the trip. This prompt will instruct the agent to consider the weather, activities, and user preferences when creating the itinerary.

Third, we will run the agent to produce the TravelPlan object, which will will refine in the following steps.

In [9]:
# Review the data structure we will use for representing a TravelPlan, which includes
# weather, activity_recommendations, and itinerary for each day of the trip.
# Our goal is to take a VacationInfo object and return a TravelPlan object.
# No changes are needed here.

class Weather(BaseModel):
    temperature: float
    temperature_unit: str
    condition: str


class Activity(BaseModel):
    activity_id: str
    name: str
    start_time: datetime.datetime
    end_time: datetime.datetime
    location: str
    description: str
    price: int
    related_interests: List[Interest]


class ActivityRecommendation(BaseModel):
    activity: Activity
    reasons_for_recommendation: List[str]


class ItineraryDay(BaseModel):
    date: datetime.date
    weather: Weather
    activity_recommendations: List[ActivityRecommendation]


class TravelPlan(BaseModel):
    city: str
    start_date: datetime.date
    end_date: datetime.date
    total_cost: int
    itinerary_days: List[ItineraryDay]

In [10]:
# Specify the Chain-of-Thought (CoT) prompt for the Itinerary Agent.
# Remember to include the following:
# [Role]: e.g. Itinerary Planning Agent
# [Task]: Explicitly or implicitly define a step-by-step process for creating the itinerary.
# [Output Format]: Respond using two sections (ANALYSIS AND FINAL OUTPUT) in the specified format.
# [Examples, optional]: Provide examples of how the output should look like.
# [Context]: Make sure to include the weather and activities data in the context, otherwise the agent won't have access to it
# and may instead hallucinate the data.
# TODO: Fill in the missing parts marked with **********

import json 
from project_lib import ChatAgent
from typing import Optional

ITINERARY_AGENT_SYSTEM_PROMPT = f"""
You are an expert travel planner for the fictional city of AgentsVille.

## Task

Create a detailed, day-by-day itinerary for the trip using the provided vacation details, weather, and activities.
Plan step-by-step: match activities to traveler interests, avoid outdoor-only activities during rain or severe weather,
ensure the total cost stays within budget, and include at least one activity per day.

## Output Format

Respond using two sections (ANALYSIS AND FINAL OUTPUT) in the following format:

    ANALYSIS:
    Provide a concise step-by-step plan (bulleted or numbered) that explains how you select activities,
    account for weather constraints, and keep total cost within budget.


    FINAL OUTPUT:

    ```json
    {json.dumps(TravelPlan.model_json_schema(), indent=2)}
    ```

## Context

Vacation Info (JSON):
{vacation_info.model_dump_json(indent=2)}

Weather Data:
{json.dumps(weather_for_dates, indent=2)}

Activities Data:
{json.dumps(activities_for_dates, indent=2)}
"""  # noqa


assert "TASK" in ITINERARY_AGENT_SYSTEM_PROMPT.upper(), "❌ ITINERARY_AGENT_SYSTEM_PROMPT should contain a 'TASK' section"
assert "OUTPUT FORMAT" in ITINERARY_AGENT_SYSTEM_PROMPT.upper(), "❌ ITINERARY_AGENT_SYSTEM_PROMPT should contain a 'OUTPUT FORMAT' section"


class ItineraryAgent(ChatAgent):
    """An agent that plans itineraries based on vacation information, weather, and activities."""
    system_prompt = ITINERARY_AGENT_SYSTEM_PROMPT

    def get_itinerary(self, vacation_info: VacationInfo, model: Optional[OpenAIModel] = None) -> TravelPlan:
        """Generates a travel itinerary based on the provided vacation information."""
        from project_lib import print_in_box
        response = (self.chat(
            user_message=vacation_info.model_dump_json(indent=2),
            add_to_messages=False,
            model=model or self.model,
        ) or "").strip()

        print_in_box(response, "Raw Response")

        # Parse the response
        json_text = response.strip()

        if "```json" in json_text:
            json_text = json_text.split("```json")[1].split("```")[0].strip()

        try:
            travel_plan = TravelPlan.model_validate_json(json_text)
            return travel_plan
        except Exception as e:
            print("Error validating the following text as TravelPlan JSON:")
            print(json_text)
            raise

itinerary_agent = ItineraryAgent(client=client, model=MODEL)


╔══════════════════════════════════════════[ ItineraryAgent - System Prompt ]══════════════════════════════════════════╗
║ You are an expert travel planner for the fictional city of AgentsVille.                                              ║
║ ## Task                                                                                                              ║
║ Create a detailed, day-by-day itinerary for the trip using the provided vacation details, weather, and activities.   ║
║ Plan step-by-step: match activities to traveler interests, avoid outdoor-only activities during rain or severe       ║
║ weather,                                                                                                             ║
║ ensure the total cost stays within budget, and include at least one activity per day.                                ║
║ ## Output Format                                                                                                     ║
║ Respond using two sections (A

In [11]:
# Generate the travel itinerary
# No changes needed here, though you can change the model to a different one if you want.

travel_plan_1 = itinerary_agent.get_itinerary(
    vacation_info=vacation_info,
    model=MODEL,  # Optionally, you can change the model here
)

if travel_plan_1 is not None:
    print("✅ Initial itinerary generated successfully. Congratulations!")


╔═══════════════════════════════════════════[ ItineraryAgent - User Prompt ]═══════════════════════════════════════════╗
║ {                                                                                                                    ║
║   "travelers": [                                                                                                     ║
║     {                                                                                                                ║
║       "name": "Yuri",                                                                                                ║
║       "age": 30,                                                                                                     ║
║       "interests": [                                                                                                 ║
║         "tennis",                                                                                                    ║
║         "cooking",           

## Evaluating the Itinerary

We've successfully created an itinerary, but how do we know if it's any good?

Now we will create some evaluation functions (sometimes called evals) to help us determine the quality of the itinerary. We will not only want our final output to be of the highest quality possible initially, but we also want to give the chance for the LLM to reflect on its own output and make improvements at a second pass.

If the itinerary does not meet all the criteria specified here, no worries! Afterwards, we will implement a feedback loop that will allow the agent to improve its plan iteratively.

In [12]:
# Helper functions for running the evaluation functions
# No change needed here.

class AgentError(Exception):
    pass


class EvaluationResults(BaseModel):
    success: bool
    failures: List[str]
    eval_functions: List[str]


def get_eval_results(vacation_info, final_output, eval_functions) -> EvaluationResults:
    """
    Evaluates the final output of the itinerary agent against a set of evaluation functions.
    Args:
        vacation_info (VacationInfo): The vacation information used to generate the itinerary.
        final_output (TravelPlan): The final output from the itinerary agent.
        eval_functions (List[callable]): A list of evaluation functions to apply.
    Returns:
        EvaluationResults: An object containing the success status, any failures, and the names of the evaluation functions used.
    """
    from project_lib import print_in_box
    if not isinstance(vacation_info, VacationInfo):
        raise ValueError("vacation_info must be an instance of VacationInfo")
    if not isinstance(final_output, TravelPlan):
        raise ValueError("final_output must be an instance of TravelPlan")
    if not isinstance(eval_functions, list) or not all(
        callable(fn) for fn in eval_functions
    ):
        raise ValueError("eval_functions must be a list of callable functions")
    eval_results = []
    for eval_fn in eval_functions:
        try:
            eval_fn(vacation_info, final_output)
        except AgentError as e:
            error_msg = str(e)
            print_in_box(error_msg, title="Evaluation Error")
            print("\n\n")

            eval_results.append(error_msg)
    return EvaluationResults(
        success=len(eval_results) == 0,
        failures=eval_results,
        eval_functions=[fn.__name__ for fn in eval_functions],
    )


In [13]:
# Basic evaluation functions
# No changes needed here.

def eval_start_end_dates_match(vacation_info: VacationInfo, final_output: TravelPlan):
    """Verifies that the arrival and departure dates in vacation_info match the start and end dates in final_output.

    Args:
        vacation_info (dict): Contains the vacation details including arrival and departure dates
        final_output (dict): Contains the itinerary details including start and end dates

    Raises:
        AgentError: If either the arrival date doesn't match the start date or the departure date doesn't match the end date
    """
    if (
        vacation_info.date_of_arrival != final_output.start_date
        or vacation_info.date_of_departure != final_output.end_date
    ):
        raise AgentError(
            f"Dates do not match: {vacation_info.date_of_arrival} != {final_output.start_date} or {vacation_info.date_of_departure} != {final_output.end_date}"
        )

    if final_output.start_date > final_output.end_date:
        raise AgentError(
            f"Start date is after end date: {final_output.start_date} > {final_output.end_date}"
        )


get_eval_results(
    vacation_info=vacation_info,
    final_output=travel_plan_1,
    eval_functions=[eval_start_end_dates_match],
)

EvaluationResults(success=True, failures=[], eval_functions=['eval_start_end_dates_match'])

In [14]:
# Evaluation functions related to the budget and total cost
# No changes needed here.


def eval_total_cost_is_accurate(vacation_info: VacationInfo, final_output: TravelPlan):
    """Verifies that the total cost stated in final_output matches the sum of all activity prices.

    Args:
        vacation_info (dict): Contains the vacation details
        final_output (dict): Contains the itinerary details including activities with prices and total cost

    Raises:
        AgentError: If the calculated total cost doesn't match the stated total cost
    """
    actual_total_cost = 0

    for itinerary_day in final_output.itinerary_days:
        for activity_recommendation in itinerary_day.activity_recommendations:
            actual_total_cost += activity_recommendation.activity.price

    stated_total_cost = int(final_output.total_cost)

    if actual_total_cost != stated_total_cost:
        raise AgentError(
            f"Stated total cost does not match calculated total cost: {actual_total_cost} != {stated_total_cost}"
        )
    
def eval_total_cost_is_within_budget(vacation_info: VacationInfo, final_output: TravelPlan):
    """Verifies that the total cost stated in final_output is within the budget specified in vacation_info.

    Args:
        vacation_info (dict): Contains the vacation details including budget
        final_output (dict): Contains the itinerary details including total cost

    Raises:
        AgentError: If the total cost exceeds the budget
    """
    stated_total_cost = int(final_output.total_cost)
    if stated_total_cost > vacation_info.budget:
        raise AgentError(
            f"Total cost exceeds budget: {stated_total_cost} > {vacation_info.budget}"
        )

get_eval_results(
    vacation_info=vacation_info,
    final_output=travel_plan_1,
    eval_functions=[eval_total_cost_is_accurate, eval_total_cost_is_within_budget],
)



╔═════════════════════════════════════════════════[ Evaluation Error ]═════════════════════════════════════════════════╗
║ Stated total cost does not match calculated total cost: 230 != 294                                                   ║
╚══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╝





EvaluationResults(success=False, failures=['Stated total cost does not match calculated total cost: 230 != 294'], eval_functions=['eval_total_cost_is_accurate', 'eval_total_cost_is_within_budget'])

In [15]:
# The final output contains copies of the activities, so we need to verify that the copies are accurate
# and not hallucinated.
# No changes needed here.

def eval_itinerary_events_match_actual_events(
    vacation_info: VacationInfo, final_output: TravelPlan
):
    """Verifies that the events listed in the itinerary match the actual events

    Args:
        vacation_info (dict): Contains the vacation details including traveler information and their interests
        final_output (dict): Contains the itinerary details including daily activities

    Raises:
        AgentError: If any traveler has no matching activities or if one traveler has more than twice
                   the number of matching activities compared to another traveler
    """
    from project_lib import call_activity_by_id_api_mocked
    event_ids_not_matching = []
    event_ids_missing = []

    for itinerary_day in final_output.itinerary_days:
        for activity_recommendation in itinerary_day.activity_recommendations:
            event_id = activity_recommendation.activity.activity_id
            # Assuming get_event_by_id is a function that retrieves the event by its ID

            reference_event = call_activity_by_id_api_mocked(event_id)

            if reference_event is None:
                event_ids_missing.append(event_id)

            elif Activity(**reference_event) != activity_recommendation.activity:
                print(
                    "---\n"
                    f"Event ID {event_id} does not match the reference event:\n"
                    f"Reference Event: {reference_event}\n"
                    f"Activity Event: {activity_recommendation.activity.model_dump()}"
                )
                event_ids_not_matching.append(event_id)
            else:
                # The event matches, so we can continue
                pass

    if event_ids_missing or event_ids_not_matching:
        raise AgentError(
            f"Event IDs missing: {event_ids_missing}\nEvent IDs not matching: {event_ids_not_matching}"
        )


get_eval_results(
    vacation_info=vacation_info,
    final_output=travel_plan_1,
    eval_functions=[eval_itinerary_events_match_actual_events],
)


EvaluationResults(success=True, failures=[], eval_functions=['eval_itinerary_events_match_actual_events'])

In [16]:
# Check that the itinerary includes at least one activity matching each traveler's interests.
# No changes needed here.

def eval_itinerary_satisfies_interests(
    vacation_info: VacationInfo, final_output: TravelPlan
):
    """Verifies that the itinerary includes activities matching each traveler's interests.

    This function checks that each traveler has at least one activity in the itinerary that matches their interests.

        Args:
        vacation_info (dict): Contains the vacation details including traveler information and their interests
        final_output (dict): Contains the itinerary details including daily activities

    Raises:
        AgentError: If any traveler has no matching activities or if one traveler has more than twice
                   the number of matching activities compared to another traveler
    """
    traveler_to_interests = {}
    traveler_to_interest_hit_counts = {}

    for traveler in vacation_info.travelers:
        traveler_to_interests[traveler.name] = traveler.interests
        traveler_to_interest_hit_counts[traveler.name] = 0

    for traveler_name, interests in traveler_to_interests.items():
        for itinerary_day in final_output.itinerary_days:
            for activity_recommendation in itinerary_day.activity_recommendations:
                # Check if the activity matches any of the traveler's interests
                matching_interests = set(traveler_to_interests[traveler_name]) & set(
                    activity_recommendation.activity.related_interests
                )

                if matching_interests:
                    traveler_to_interest_hit_counts[traveler_name] += 1
                    print(
                        f"✅ Traveler {traveler_name} has a match with interest {matching_interests} at {activity_recommendation.activity.name}"
                    )

    travelers_with_no_interest_hits = [
        traveler
        for traveler, interest_hit_count in traveler_to_interest_hit_counts.items()
        if interest_hit_count == 0
    ]

    # If any of the travelers have 0 matches, raise an error
    if travelers_with_no_interest_hits:
        raise AgentError(
            f"Travelers {travelers_with_no_interest_hits} has no matches with the itinerary."
        )


get_eval_results(
    vacation_info=vacation_info,
    final_output=travel_plan_1,
    eval_functions=[eval_itinerary_satisfies_interests],
)


✅ Traveler Yuri has a match with interest {technology} at Tech & Math Brunch Meetup
✅ Traveler Yuri has a match with interest {comedy} at Evening Comedy Showcase
✅ Traveler Yuri has a match with interest {technology} at Future Apps Tech Meetup
✅ Traveler Yuri has a match with interest {technology} at Robotics Workshop: Build & Code
✅ Traveler Yuri has a match with interest {technology} at Math & Science Micro-Lectures
✅ Traveler Yuri has a match with interest {technology} at Makerspace Open Lab
✅ Traveler Yuri has a match with interest {cooking} at Plant-Powered Cooking Class
✅ Traveler Yuri has a match with interest {tennis} at Sunrise Tennis Rally
✅ Traveler Yuri has a match with interest {cooking} at Comfort Cooking & Nutrition Lab
✅ Traveler Hiro has a match with interest {reading} at Reading Salon & City Stories
✅ Traveler Hiro has a match with interest {art} at Photography Studio: Light & Composition
✅ Traveler Hiro has a match with interest {art, reading} at Creative Writing Caf

EvaluationResults(success=True, failures=[], eval_functions=['eval_itinerary_satisfies_interests'])

In [17]:
# Use an LLM to determine whether an event should be avoided due to weather conditions.
# TODO: Fill in the missing parts marked with **********

ACTIVITY_AND_WEATHER_ARE_COMPATIBLE_SYSTEM_PROMPT = """
You are a travel safety reviewer who decides if an activity is compatible with the given weather.

## Task
Given an activity name/description and a weather condition, decide whether the activity should proceed.
If information is insufficient, assume the activity IS_COMPATIBLE. If the activity is outdoor-only and weather is bad
(rain, storm, heavy wind), mark it IS_INCOMPATIBLE. Consider backup/indoor options mentioned in the description.

## Output format

    REASONING:
    One or two sentences explaining the decision.

    FINAL ANSWER:
    [IS_COMPATIBLE, IS_INCOMPATIBLE]

## Examples
`
Activity: Riverside Picnic
Description: Outdoor-only picnic by the river. No indoor backup.
Weather Condition: Rainy

REASONING:
Outdoor-only activity with rain and no backup.
FINAL ANSWER:
IS_INCOMPATIBLE
`

`
Activity: Modern Art Museum Tour
Description: Guided indoor museum tour. Great for rainy days.
Weather Condition: Rainy

REASONING:
Indoor activity, weather does not prevent attendance.
FINAL ANSWER:
IS_COMPATIBLE
""".strip()



def eval_activities_and_weather_are_compatible(
    vacation_info: VacationInfo, final_output: TravelPlan
):
    """Verifies that no outdoor-only activities are scheduled during inclement weather conditions.

    Args:
        vacation_info (dict): Contains the vacation details
        final_output (dict): Contains the itinerary details including daily activities and weather conditions

    Raises:
        AgentError: If any outdoor activities are scheduled during weather conditions that could ruin them
    """
    from project_lib import do_chat_completion

    activities_that_are_incompatible = []

    for itinerary_day in final_output.itinerary_days:
        weather_condition = itinerary_day.weather.condition

        for activity_recommendation in itinerary_day.activity_recommendations:
            resp = do_chat_completion(
                messages=[
                    {
                        "role": "system",
                        "content": ACTIVITY_AND_WEATHER_ARE_COMPATIBLE_SYSTEM_PROMPT,
                    },
                    {
                        "role": "user",
                        "content": f"Activity: {activity_recommendation.activity.name}\nDescription: {activity_recommendation.activity.description}\nWeather Condition: {weather_condition}",
                    },
                ],
                client=client,
                # This is a high-frequency use case, so we use a fast and cheap model.
                model=OpenAIModel.GPT_41_NANO,
            )
    


            if "IS_COMPATIBLE" in (resp or ""):
                is_compatible = True
            elif "IS_INCOMPATIBLE" in (resp or ""):
                is_compatible = False
            else:
                raise RuntimeError(
                    f"Unexpected response from the model: {resp}. Expected 'IS_COMPATIBLE' or 'IS_INCOMPATIBLE'."
                )

            if is_compatible:
                print(
                    f"✅ Activity {activity_recommendation.activity.name} (on {itinerary_day.date}) and weather '{weather_condition}' are compatible."
                )

            else:
                activities_that_are_incompatible.append(
                    activity_recommendation.activity.name
                )
                print(
                    f"❌ Activity {activity_recommendation.activity.name} (on {itinerary_day.date}) and weather '{weather_condition}' are incompatible."
                )

    if activities_that_are_incompatible:
        raise AgentError(
            f"Activities that may be ruined by inclement weather: {activities_that_are_incompatible}"
        )


eval_results = get_eval_results(
    vacation_info=vacation_info,
    final_output=travel_plan_1,
    eval_functions=[
        eval_activities_and_weather_are_compatible
    ],
)

eval_results

✅ Activity Tech & Math Brunch Meetup (on 2026-03-01) and weather 'partly cloudy' are compatible.
✅ Activity Evening Comedy Showcase (on 2026-03-01) and weather 'partly cloudy' are compatible.
✅ Activity Future Apps Tech Meetup (on 2026-03-02) and weather 'sunny' are compatible.
✅ Activity Robotics Workshop: Build & Code (on 2026-03-03) and weather 'rainy' are compatible.
✅ Activity Reading Salon & City Stories (on 2026-03-04) and weather 'cloudy' are compatible.
✅ Activity Photography Studio: Light & Composition (on 2026-03-04) and weather 'cloudy' are compatible.
✅ Activity Math & Science Micro-Lectures (on 2026-03-05) and weather 'thunderstorm' are compatible.
✅ Activity Creative Writing Cafe (on 2026-03-05) and weather 'thunderstorm' are compatible.
✅ Activity Makerspace Open Lab (on 2026-03-06) and weather 'partly cloudy' are compatible.
✅ Activity Plant-Powered Cooking Class (on 2026-03-06) and weather 'partly cloudy' are compatible.
✅ Activity Sunrise Tennis Rally (on 2026-03-07)

EvaluationResults(success=True, failures=[], eval_functions=['eval_activities_and_weather_are_compatible'])

In [18]:
# Run all of the evaluation functions again
# No changes needed here.

ALL_EVAL_FUNCTIONS = [
    eval_start_end_dates_match,
    eval_total_cost_is_accurate,
    eval_itinerary_events_match_actual_events,
    eval_itinerary_satisfies_interests,
    eval_total_cost_is_within_budget,
    eval_activities_and_weather_are_compatible,
]

eval_results = get_eval_results(
    vacation_info=vacation_info,
    final_output=travel_plan_1,
    eval_functions=ALL_EVAL_FUNCTIONS,
)

eval_results.model_dump()


╔═════════════════════════════════════════════════[ Evaluation Error ]═════════════════════════════════════════════════╗
║ Stated total cost does not match calculated total cost: 230 != 294                                                   ║
╚══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╝



✅ Traveler Yuri has a match with interest {technology} at Tech & Math Brunch Meetup
✅ Traveler Yuri has a match with interest {comedy} at Evening Comedy Showcase
✅ Traveler Yuri has a match with interest {technology} at Future Apps Tech Meetup
✅ Traveler Yuri has a match with interest {technology} at Robotics Workshop: Build & Code
✅ Traveler Yuri has a match with interest {technology} at Math & Science Micro-Lectures
✅ Traveler Yuri has a match with interest {technology} at Makerspace Open Lab
✅ Traveler Yuri has a match with interest {cooking} at Plant-Powered Cooking Class
✅ Traveler Yuri has a match with interest {tennis}

{'success': False,
 'failures': ['Stated total cost does not match calculated total cost: 230 != 294'],
 'eval_functions': ['eval_start_end_dates_match',
  'eval_total_cost_is_accurate',
  'eval_itinerary_events_match_actual_events',
  'eval_itinerary_satisfies_interests',
  'eval_total_cost_is_within_budget',
  'eval_activities_and_weather_are_compatible']}

## Defining the Tools

Our ItineraryRevisionAgent will be a ReAct-based agent that will use tools to:
- Evaluate/Re-evaluate the itinerary
- Use a calculator since LLMs sometimes struggle with arithmetic
- Call the activities API to get more information about activities
- Return the final itinerary


In [19]:
# Helper function to generate tool descriptions from function docstrings
# No changes needed here.

def get_tool_descriptions_string(fns):
    """Generates a tool description from a function's docstring.
    Args:
        fns (list): List of functions to generate descriptions for.
    Returns:
        str: A formatted string containing the function names and their descriptions."""
    resp = ""
    for fn in fns:
        function_name = fn.__name__
        function_doc = fn.__doc__ or "No description provided."

        resp += f"* `{function_name}`: {function_doc}\n"

    return resp

In [20]:
# Define the calculator tool that evaluates mathematical expressions.
# No changes needed here.

def calculator_tool(input_expression) -> float:
    """Evaluates a mathematical expression and returns the result as a float.

    Args:
        input_expression (str): A string containing a valid mathematical expression to evaluate.

    Returns:
        float: The result of the evaluated expression.

    Example:
        >>> calculator_tool("1 + 1")
        2.0
    """
    import numexpr as ne
    return float(ne.evaluate(input_expression))


assert calculator_tool("1 + 1") == 2.0

print(get_tool_descriptions_string([calculator_tool]))

* `calculator_tool`: Evaluates a mathematical expression and returns the result as a float.

Args:
    input_expression (str): A string containing a valid mathematical expression to evaluate.

Returns:
    float: The result of the evaluated expression.

Example:
    >>> calculator_tool("1 + 1")
    2.0




In [21]:
# Tool to fetch activities for a given date and city.
# TODO: Fill in the missing parts marked with **********

def get_activities_by_date_tool(date: str, city: str) -> List[dict]:
    """Returns the list of available activities for a given date and city.

    Args:
        date (str): Date string in YYYY-MM-DD format.
        city (str): City name to search activities in (e.g., "AgentsVille").

    Returns:
        List[dict]: A list of activity objects (validated against the Activity model) serialized as dictionaries.

    Example:
        >>> get_activities_by_date_tool("2025-06-10", "AgentsVille")
        [{"activity_id": "...", "name": "...", ...}]
    """
    from project_lib import call_activities_api_mocked
    resp = call_activities_api_mocked(date=date, city=city)

    return [Activity.model_validate(activity).model_dump() for activity in resp]



assert len(get_activities_by_date_tool("2025-06-10", "AgentsVille")) > 0

print(get_tool_descriptions_string([get_activities_by_date_tool]))

* `get_activities_by_date_tool`: Returns the list of available activities for a given date and city.

Args:
    date (str): Date string in YYYY-MM-DD format.
    city (str): City name to search activities in (e.g., "AgentsVille").

Returns:
    List[dict]: A list of activity objects (validated against the Activity model) serialized as dictionaries.

Example:
    >>> get_activities_by_date_tool("2025-06-10", "AgentsVille")
    [{"activity_id": "...", "name": "...", ...}]




In [22]:
# Tool to run all evaluation functions on a travel plan.
# No changes needed here.

def run_evals_tool(travel_plan: TravelPlan) -> dict:
    """Runs all evaluation tools on the provided travel plan and vacation info.

    Args:
        travel_plan (TravelPlan): The travel plan to evaluate.

    Returns:
        EvaluationResults: The results of the evaluations.
    """
    if isinstance(travel_plan, dict):
        travel_plan = TravelPlan.model_validate(travel_plan)

    resp = get_eval_results(
        vacation_info=vacation_info,
        final_output=travel_plan,
        eval_functions=ALL_EVAL_FUNCTIONS,
    )
    return {
        # Show the success status and any failures
        "success": resp.success,
        "failures": resp.failures,
    }


print(get_tool_descriptions_string([run_evals_tool]))

* `run_evals_tool`: Runs all evaluation tools on the provided travel plan and vacation info.

Args:
    travel_plan (TravelPlan): The travel plan to evaluate.

Returns:
    EvaluationResults: The results of the evaluations.




In [23]:
# Let's double check that the tool works as expected.
# You should see the same results as before
run_evals_tool(travel_plan=travel_plan_1)


╔═════════════════════════════════════════════════[ Evaluation Error ]═════════════════════════════════════════════════╗
║ Stated total cost does not match calculated total cost: 230 != 294                                                   ║
╚══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╝



✅ Traveler Yuri has a match with interest {technology} at Tech & Math Brunch Meetup
✅ Traveler Yuri has a match with interest {comedy} at Evening Comedy Showcase
✅ Traveler Yuri has a match with interest {technology} at Future Apps Tech Meetup
✅ Traveler Yuri has a match with interest {technology} at Robotics Workshop: Build & Code
✅ Traveler Yuri has a match with interest {technology} at Math & Science Micro-Lectures
✅ Traveler Yuri has a match with interest {technology} at Makerspace Open Lab
✅ Traveler Yuri has a match with interest {cooking} at Plant-Powered Cooking Class
✅ Traveler Yuri has a match with interest {tennis}

{'success': False,
 'failures': ['Stated total cost does not match calculated total cost: 230 != 294']}

In [24]:
# A tool to return the final travel plan
# No changes needed here.

def final_answer_tool(final_output: TravelPlan) -> TravelPlan:
    """Returns the final travel plan

    Args:
        final_output (TravelPlan): The final travel plan to return.

    Returns:
        TravelPlan: The final travel plan.
    """
    return final_output


print(get_tool_descriptions_string([final_answer_tool]))

* `final_answer_tool`: Returns the final travel plan

Args:
    final_output (TravelPlan): The final travel plan to return.

Returns:
    TravelPlan: The final travel plan.




In [25]:
# List of all tools available for the agent
# No changes needed here.

ALL_TOOLS = [
    calculator_tool,
    get_activities_by_date_tool,
    run_evals_tool,
    final_answer_tool,
]
print(get_tool_descriptions_string(ALL_TOOLS))

* `calculator_tool`: Evaluates a mathematical expression and returns the result as a float.

Args:
    input_expression (str): A string containing a valid mathematical expression to evaluate.

Returns:
    float: The result of the evaluated expression.

Example:
    >>> calculator_tool("1 + 1")
    2.0

* `get_activities_by_date_tool`: Returns the list of available activities for a given date and city.

Args:
    date (str): Date string in YYYY-MM-DD format.
    city (str): City name to search activities in (e.g., "AgentsVille").

Returns:
    List[dict]: A list of activity objects (validated against the Activity model) serialized as dictionaries.

Example:
    >>> get_activities_by_date_tool("2025-06-10", "AgentsVille")
    [{"activity_id": "...", "name": "...", ...}]

* `run_evals_tool`: Runs all evaluation tools on the provided travel plan and vacation info.

Args:
    travel_plan (TravelPlan): The travel plan to evaluate.

Returns:
    EvaluationResults: The results of the evaluati

## The ItineraryRevisionAgent

The ItineraryRevisionAgent will
* take initial feedback from the user about the itinerary and
* use the tools defined above

to refine original itinerary iteratively using a ReAct-based approach.

In [26]:
# Get the traveler's feedback and create a new evaluation function to check if the feedback was incorporated.
# No changes needed here.

TRAVELER_FEEDBACK = "I want to have at least two activities per day."


def eval_traveler_feedback_is_incorporated(
    vacation_info: VacationInfo, final_output: TravelPlan
):
    """Checks if the traveler's feedback was incorporated into the revised travel plan.

    Args:
        vacation_info (VacationInfo): The vacation information.
        final_output (TravelPlan): The revised travel plan.

    Raises:
        AgentError: If the traveler's feedback was not successfully incorporated.
    """

    agent = ChatAgent(
        system_prompt="""You are an expert in evaluating whether a travel plan incorporates traveler feedback.

    ## Output Format

    Respond using two sections (ANALYSIS AND FINAL OUTPUT) in the following format:

        ANALYSIS:
        * [step-by-step analysis]


        FINAL OUTPUT:
        [FULLY_INCORPORATED, PARTIALLY_INCORPORATED, NOT_INCORPORATED, or UNKNOWN]
        REASON: [reasoning for the final output]

    """,
        client=client,
        model=OpenAIModel.GPT_41,  # Use a powerful model for checking traveler feedback
    )

    resp = agent.chat(
        f"""Traveler Feedback: {TRAVELER_FEEDBACK}
    Revised Travel Plan: {final_output.model_dump_json()}
    """,
    )
    if "FINAL OUTPUT:" not in resp:
        raise RuntimeError(
            f"Unexpected response from the model: {resp}. Expected 'FINAL OUTPUT:'."
        )
    if "FULLY_INCORPORATED" not in resp:
        final_output = resp.split("FINAL OUTPUT:")[-1].strip()
        raise AgentError(
            f"Traveler feedback was not successfully incorporated into the revised travel plan. Response: {final_output}"
        )

ALL_EVAL_FUNCTIONS = [
    eval_start_end_dates_match,
    eval_total_cost_is_accurate,
    eval_itinerary_events_match_actual_events,
    eval_itinerary_satisfies_interests,
    eval_total_cost_is_within_budget,
    eval_activities_and_weather_are_compatible,
    eval_traveler_feedback_is_incorporated,  # Add this new evaluation
]

get_eval_results(
    vacation_info=vacation_info,
    final_output=travel_plan_1,
    eval_functions=ALL_EVAL_FUNCTIONS,
)


╔═════════════════════════════════════════════════[ Evaluation Error ]═════════════════════════════════════════════════╗
║ Stated total cost does not match calculated total cost: 230 != 294                                                   ║
╚══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╝



✅ Traveler Yuri has a match with interest {technology} at Tech & Math Brunch Meetup
✅ Traveler Yuri has a match with interest {comedy} at Evening Comedy Showcase
✅ Traveler Yuri has a match with interest {technology} at Future Apps Tech Meetup
✅ Traveler Yuri has a match with interest {technology} at Robotics Workshop: Build & Code
✅ Traveler Yuri has a match with interest {technology} at Math & Science Micro-Lectures
✅ Traveler Yuri has a match with interest {technology} at Makerspace Open Lab
✅ Traveler Yuri has a match with interest {cooking} at Plant-Powered Cooking Class
✅ Traveler Yuri has a match with interest {tennis}

EvaluationResults(success=False, failures=['Stated total cost does not match calculated total cost: 230 != 294', 'Traveler feedback was not successfully incorporated into the revised travel plan. Response: PARTIALLY_INCORPORATED\nREASON: The revised plan provides at least two activities on most days but fails to do so on March 2, March 3, and March 8. Therefore, the traveler\'s feedback of having "at least two activities per day" is only partially incorporated.'], eval_functions=['eval_start_end_dates_match', 'eval_total_cost_is_accurate', 'eval_itinerary_events_match_actual_events', 'eval_itinerary_satisfies_interests', 'eval_total_cost_is_within_budget', 'eval_activities_and_weather_are_compatible', 'eval_traveler_feedback_is_incorporated'])

In [27]:
# Define the ReAct system prompt for the Itinerary Revision Agent.
# Remember, the output format should include a THOUGHT and a single ACTION (tool call using JSON format).
# Then the Python code will invoke the tool and add an OBSERVATION message to the chat history.
# NOTE: The tool call format should be:
# {"tool_name": "[tool_name]", "arguments": {"arg1": "value1", ...}}

# TODO: Fill in the missing parts marked with **********
from project_lib import print_in_box

ITINERARY_REVISION_AGENT_SYSTEM_PROMPT = f"""
You are an itinerary revision agent that improves a draft travel plan using tools and feedback.

## Task

Revise the itinerary to satisfy traveler feedback (at least 2 activities per day) and all evaluation criteria.
If the budget is tight, prioritize the lowest-cost options and keep exactly two activities per day unless a third
activity is needed to satisfy a missing interest.
You must invoke run_evals_tool to get feedback on the current plan, make improvements, and then run run_evals_tool again
before calling final_answer_tool.

Process (mandatory):
1) First ACTION must be run_evals_tool with the current TravelPlan.
2) If there are failures, revise the TravelPlan in your next reasoning step.
3) When you need data for a date, call get_activities_by_date_tool.
   After any get_activities_by_date_tool call, your next ACTION must revise the plan and then run run_evals_tool
   (use calculator_tool first if you changed activities).
4) After you revise the plan, call run_evals_tool again and include the FULL revised TravelPlan in the arguments.
5) If run_evals_tool returns success=true, immediately call final_answer_tool with the same TravelPlan and stop.

Cost accuracy rules:
- Always recalculate total_cost using calculator_tool whenever you add/remove/replace activities.
- Compute total_cost as the exact sum of all activity prices included in itinerary_days.
- Do not estimate or round costs; use calculator_tool and update TravelPlan.total_cost with its result.
- If the eval reports a total cost mismatch, your next ACTION must call calculator_tool to recompute it.
- After calling calculator_tool, you must update TravelPlan.total_cost and then call run_evals_tool next.

Important tool-call requirements:
- run_evals_tool requires the FULL TravelPlan JSON in arguments as {{"travel_plan": <TravelPlan>}}.
- final_answer_tool requires the FULL TravelPlan JSON in arguments as {{"final_output": <TravelPlan>}}.

Use get_activities_by_date_tool to fetch activities for specific dates when you need to replace or add activities.

## Available Tools

{get_tool_descriptions_string(ALL_TOOLS)}

## Output Format

    THOUGHT:
    Provide your reasoning steps for the next action.

    ACTION:
    Use a single tool call in the exact JSON format:
    {{"tool_name": "[tool_name]", "arguments": {{"arg1": "value1", "arg2": "value2"}}}}
    Only one ACTION per message.

## Context

Traveler Feedback:
{TRAVELER_FEEDBACK}

Vacation Info (JSON):
{vacation_info.model_dump_json(indent=2)}

TravelPlan Schema:
{json.dumps(TravelPlan.model_json_schema(), indent=2)}

You must follow the THINK → ACTION → OBSERVATION loop. After each OBSERVATION, produce a new THOUGHT and ACTION.
Before calling final_answer_tool, you must run run_evals_tool and ensure all evaluation criteria pass.
Exit only by calling final_answer_tool with the final TravelPlan once all evaluations pass.
"""  # noqa


class ItineraryRevisionAgent(ChatAgent):
    system_prompt = ITINERARY_REVISION_AGENT_SYSTEM_PROMPT
    tools = ALL_TOOLS

    def get_observation_string(self, tool_call_obj):
        """Extracts the observation from the thought-action response."""

        if "tool_name" not in tool_call_obj:
            return "OBSERVATION: No tool name specified.", None

        if "arguments" not in tool_call_obj:
            return "OBSERVATION: No arguments specified.", None

        # If the arguments are not a dictionary, state the error
        if not isinstance(tool_call_obj["arguments"], dict):
            return (
                f"OBSERVATION: Arguments should be a dictionary, got {type(tool_call_obj['arguments'])} instead.",
                None,
            )

        # If the tool name is not a string, state the error
        if not isinstance(tool_call_obj["tool_name"], str):
            return (
                f"OBSERVATION: Tool name should be a string, got {type(tool_call_obj['tool_name'])} instead.",
                None,
            )

        tool_name = tool_call_obj["tool_name"]
        arguments = tool_call_obj["arguments"]

        tool_fn = None

        for tool in self.tools:
            if tool.__name__ == tool_name:
                tool_fn = tool
                break

        if tool_fn is None:
            return f"OBSERVATION: Unknown tool name '{tool_name}' in action string.", None

        try:
            tool_response = tool_fn(**arguments)
            return (
                f"OBSERVATION: Tool {tool_name} called successfully with response: {tool_response}",
                tool_response,
            )
        except Exception as e:
            return f"OBSERVATION: Error occurred while calling tool {tool_name}: {e}", None

    def run_react_cycle(
        self, original_travel_plan: TravelPlan, max_steps: int = 10, model: Optional[OpenAIModel] = None, client = None,
    ) -> TravelPlan:
        """Runs the ReAct cycle to revise the itinerary based on the evaluation results."""
        from json_repair import repair_json

        # Provide the original travel plan to revise
        self.add_message(
            role="user",
            content=f"Here is the itinerary for revision:\n{original_travel_plan.model_dump_json()}",
        )
        resp = None
        last_eval_result = None
        last_eval_failures = None
        last_eval_plan_payload = None

        # Run the ReAct cycle for a maximum number of steps
        for step in range(max_steps):
            # Get the thought-action response from the agent
            resp = self.get_response(model=model, client=client) or ""

            # If there is no action, report it to the LLM and continue
            if "ACTION:" not in resp:
                self.add_message(role="user", content="No action found in response.")
                continue

            action_string = resp.split("ACTION:")[1].strip()

            # Parse the tool call JSON from the action string
            try:
                # Fix any JSON formatting issues. e.g. missing closing braces, etc.
                action_string = repair_json(action_string)
                tool_call_obj = json.loads(action_string)
            except json.JSONDecodeError:
                print(f"Invalid JSON in action string: {action_string}")
                self.add_message(
                    role="user",
                    content=f"Invalid JSON in action string: {action_string}",
                )
                continue

            tool_name = tool_call_obj.get("tool_name", None)

            # If the final answer tool is called, validate and return the final travel plan
            if tool_name == "final_answer_tool":
                try:
                    new_travel_plan = TravelPlan.model_validate(
                        tool_call_obj["arguments"].get("final_output", tool_call_obj["arguments"])
                    )
                    return new_travel_plan
                except Exception as e:
                    self.add_message(
                        role="user", content=f"Error validating final answer: {e}"
                    )
                    continue

            # For all other tools, execute the tool call and add the observation
            else:
                # Add the 
                observation_string, tool_response = self.get_observation_string(
                    tool_call_obj=tool_call_obj
                )
                self.add_message(role="user", content=observation_string)

                if tool_name == "run_evals_tool" and isinstance(tool_response, dict):
                    last_eval_result = tool_response
                    last_eval_failures = tool_response.get("failures")
                    last_eval_plan_payload = tool_call_obj["arguments"].get(
                        "travel_plan", tool_call_obj["arguments"]
                    )
                    if tool_response.get("success") is True:
                        try:
                            new_travel_plan = TravelPlan.model_validate(last_eval_plan_payload)
                            return new_travel_plan
                        except Exception as e:
                            self.add_message(
                                role="user",
                                content=f"Error validating travel_plan after successful evals: {e}",
                            )
                            continue

        if last_eval_result is None:
            try:
                fallback_result = run_evals_tool(travel_plan=original_travel_plan)
                last_eval_result = fallback_result
                last_eval_failures = fallback_result.get("failures")
                last_eval_plan_payload = original_travel_plan.model_dump()
            except Exception:
                pass

        diagnostics = ""
        if last_eval_result is not None:
            if last_eval_result.get("success") is True:
                diagnostics = "Últimas evals indicaram sucesso, mas o ciclo não finalizou."
            else:
                diagnostics = f"Critérios não aprovados nas últimas evals: {last_eval_failures}"

        raise RuntimeError(
            f"ReAct cycle did not complete within {max_steps} steps. {diagnostics} Last response: {resp}"
        )


def debug_react_cycle_with_responses(
    original_travel_plan: TravelPlan,
    responses: list[str],
    max_steps: int = 10,
) -> TravelPlan:
    """Runs a deterministic ReAct cycle using pre-baked responses (no LLM calls)."""
    agent = ItineraryRevisionAgent()
    response_iter = iter(responses)

    def _fake_get_response(*args, **kwargs):
        return next(response_iter, "")

    agent.get_response = _fake_get_response  # type: ignore[method-assign]
    return agent.run_react_cycle(
        original_travel_plan=original_travel_plan,
        max_steps=max_steps,
        model=None,
        client=None,
    )

# Instantiate the Itinerary Revision Agent
itinerary_revision_agent = ItineraryRevisionAgent()

# Let's get a single THOUGHT/ACTION response back to check that the agent is working as expected.
resp = itinerary_revision_agent.chat(
    user_message=f"Here is the itinerary for revision: {travel_plan_1.model_dump_json(indent=2)}",
    add_to_messages=False,
    model=MODEL,
    client=client,
) or ""

print_in_box(resp, "Raw Response")
# Check for THOUGHT
if "THOUGHT:" in resp:
    print("✅ `THOUGHT:` found in raw the response, as expected.")
else:
    print("❌ Expected `THOUGHT:` in raw the response. Please check the system prompt (output format).")
# Check for ACTION
if "ACTION:" in resp:
    print("✅ `ACTION:` found in raw the response, as expected.")
else:
    print("❌ Expected `ACTION:` in raw the response. Please check the system prompt (output format).")
if "\"tool_name\"" in resp:
    print("✅ `\"tool_name\":` found in raw the response, as expected.")
else:
    print("❌ Expected `\"tool_name\":` in raw the response. Please check the system prompt (output format).")



╔══════════════════════════════════════[ ItineraryRevisionAgent - System Prompt ]══════════════════════════════════════╗
║ You are an itinerary revision agent that improves a draft travel plan using tools and feedback.                      ║
║ ## Task                                                                                                              ║
║ Revise the itinerary to satisfy traveler feedback (at least 2 activities per day) and all evaluation criteria.       ║
║ If the budget is tight, prioritize the lowest-cost options and keep exactly two activities per day unless a third    ║
║ activity is needed to satisfy a missing interest.                                                                    ║
║ You must invoke run_evals_tool to get feedback on the current plan, make improvements, and then run run_evals_tool   ║
║ again                                                                                                                ║
║ before calling final_answer_t

In [28]:
# Now let's run the ReAct cycle multiple times to get the revised itinerary.
# Note: If this takes more than a few minutes, then the agent may be stuck in a loop.
# Examine the traces to understand where it is failing and see if adjusting the system prompt helps.
# Since LLMs are stochastic, you will get different results each time you run this cell.
# No changes needed here.

itinerary_revision_agent = ItineraryRevisionAgent()
travel_plan_2 = itinerary_revision_agent.run_react_cycle(
    original_travel_plan=travel_plan_1, max_steps=15,
    model=MODEL,
    client=client,
)

print("✅ Revised itinerary generated successfully. Congratulations!")



╔══════════════════════════════════════[ ItineraryRevisionAgent - System Prompt ]══════════════════════════════════════╗
║ You are an itinerary revision agent that improves a draft travel plan using tools and feedback.                      ║
║ ## Task                                                                                                              ║
║ Revise the itinerary to satisfy traveler feedback (at least 2 activities per day) and all evaluation criteria.       ║
║ If the budget is tight, prioritize the lowest-cost options and keep exactly two activities per day unless a third    ║
║ activity is needed to satisfy a missing interest.                                                                    ║
║ You must invoke run_evals_tool to get feedback on the current plan, make improvements, and then run run_evals_tool   ║
║ again                                                                                                                ║
║ before calling final_answer_t

In [29]:
# Last let's double check that the revised travel plan passes all evaluation functions.
# No changes needed here.

eval_results_2 = get_eval_results(
    vacation_info=vacation_info,
    final_output=travel_plan_2,
    eval_functions=ALL_EVAL_FUNCTIONS,
)

assert eval_results_2.success, f"❌ Read the traces above and modify the system prompt.\n\nFailures: {eval_results_2.failures}"

print("✅ All evaluation functions passed successfully for the revised travel plan.")

eval_results_2

✅ Traveler Yuri has a match with interest {comedy} at Evening Comedy Showcase
✅ Traveler Yuri has a match with interest {tennis} at Indoor Tennis Skills Lab
✅ Traveler Yuri has a match with interest {cooking} at Rainy Day Cooking & Nutrition Class
✅ Traveler Yuri has a match with interest {technology} at Math & Science Micro-Lectures
✅ Traveler Yuri has a match with interest {technology} at Makerspace Open Lab
✅ Traveler Yuri has a match with interest {tennis} at Sunrise Tennis Rally
✅ Traveler Yuri has a match with interest {cooking} at Comfort Cooking & Nutrition Lab
✅ Traveler Hiro has a match with interest {art} at Sunrise Photography Walk
✅ Traveler Hiro has a match with interest {reading} at Reading Salon & City Stories
✅ Traveler Hiro has a match with interest {art} at Photography Studio: Light & Composition
✅ Traveler Hiro has a match with interest {art, reading} at Creative Writing Cafe
✅ Traveler Hiro has a match with interest {art, theatre} at Backstage Theatre Q&A
✅ Travele

EvaluationResults(success=True, failures=[], eval_functions=['eval_start_end_dates_match', 'eval_total_cost_is_accurate', 'eval_itinerary_events_match_actual_events', 'eval_itinerary_satisfies_interests', 'eval_total_cost_is_within_budget', 'eval_activities_and_weather_are_compatible', 'eval_traveler_feedback_is_incorporated'])

In [30]:
# Show the final travel plan in a readable format.
# No changes needed here.

from IPython.display import display

for itinerary_day in travel_plan_2.itinerary_days:
    print(f"Date: {itinerary_day.date}")
    print(
        f"Weather: {itinerary_day.weather.condition} ({itinerary_day.weather.temperature}°{itinerary_day.weather.temperature_unit})"
    )

    activities_df = pd.DataFrame(
        [
            activity_recommendation.activity.model_dump()
            for activity_recommendation in itinerary_day.activity_recommendations
        ]
    )
    display(activities_df)

Date: 2026-03-01
Weather: partly cloudy (22.0°celsius)


,activity_id,name,start_time,end_time,location,description,price,related_interests
0,event-2026-03-01-1,Riverfront Soccer Skills Clinic,2026-03-01 12:00:00,2026-03-01 13:30:00,"Riverside Fields, AgentsVille","Join a friendly soccer clinic focused on drills, footwork, and teamwork by the river. This outdoor session includes a warm-up and short scrimmage; in case of rain, we move to the nearby Community Sports Hall.",15,"[soccer, fitness]"
1,event-2026-03-01-3,Evening Comedy Showcase,2026-03-01 19:00:00,2026-03-01 20:30:00,"Laughline Theater, AgentsVille",Relax indoors with a curated lineup of local comedians. A lighthearted night of stand-up and sketches perfect for comedy fans.,15,[comedy]


Date: 2026-03-02
Weather: sunny (24.0°celsius)


,activity_id,name,start_time,end_time,location,description,price,related_interests
0,event-2026-03-02-1,AgentsVille Soccer Friendly,2026-03-02 12:00:00,2026-03-02 13:30:00,"Sunfield Park Pitch, AgentsVille","Join a casual outdoor soccer match with locals. All skill levels welcome for a fun, energetic mid-day game.",10,"[soccer, fitness]"
1,event-2026-03-02-0,Sunrise Photography Walk,2026-03-02 08:00:00,2026-03-02 10:00:00,"Old Town Promenade, AgentsVille",Capture golden morning light on a guided photography walk through historic streets and scenic viewpoints. Outdoor event with indoor gallery stop included.,15,"[photography, art]"


Date: 2026-03-03
Weather: rainy (20.0°celsius)


...

,activity_id,name,start_time,end_time,location,description,price,related_interests
0,event-2026-03-06-1,Makerspace Open Lab,2026-03-06 12:30:00,2026-03-06 14:30:00,"Makerspace Hub, AgentsVille","Drop in for an indoor makerspace session with 3D printers, electronics kits, and guided programming help.",20,"[technology, programming]"
1,event-2026-03-06-3,Backstage Theatre Q&A,2026-03-06 19:00:00,2026-03-06 20:30:00,"Grand Theatre, AgentsVille",Go behind the scenes with theatre artists for an indoor Q&A and short rehearsal preview.,16,"[theatre, art]"


Date: 2026-03-07
Weather: clear (23.0°celsius)


,activity_id,name,start_time,end_time,location,description,price,related_interests
0,event-2026-03-07-1,Sunrise Tennis Rally,2026-03-07 12:00:00,2026-03-07 13:30:00,"Grand Courts at Sunfield Park, AgentsVille",Outdoor tennis rally with friendly drills and round-robin games. Great for tennis and fitness fans.,15,"[tennis, fitness]"
1,event-2026-03-07-2,Live Music Picnic,2026-03-07 15:00:00,2026-03-07 17:00:00,"Echo Gardens Amphitheater, AgentsVille",Enjoy outdoor live music performances with picnic blankets and casual dancing. A relaxed afternoon for music lovers.,14,"[music, dancing]"


Date: 2026-03-08
Weather: rainy (17.0°celsius)


,activity_id,name,start_time,end_time,location,description,price,related_interests
0,event-2026-03-08-2,Comfort Cooking & Nutrition Lab,2026-03-08 15:00:00,2026-03-08 17:00:00,"Culinary Lab Studio, AgentsVille",Indoor cooking session focused on nourishing comfort foods and nutrition basics for rainy days.,22,"[nutrition, cooking]"
1,event-2026-03-08-0,Indoor Soccer Futsal,2026-03-08 09:00:00,2026-03-08 10:30:00,"Community Sports Hall, AgentsVille",Stay active indoors with a fast-paced futsal session for soccer fans of all levels.,12,"[soccer, fitness]"


## And, just for fun!

In [31]:
# And finally, just for fun, let's narrate the trip.
# No changes needed here.

from project_lib import narrate_my_trip

narrate_my_trip(
    vacation_info=vacation_info,
    itinerary=travel_plan_2,
    client=client,
    model=MODEL,  # Optionally, you can change the model here
)


Welcome to your upcoming trip to AgentsVille from March 1st to March 8th, 2026! The group consists of three travelers: Yuri, age 30, whose interests include tennis, cooking, comedy, and technology; Hiro, age 25, who enjoys reading, music, theatre, and art; and Kaoru Kishimoto, age 54, with interests in math, programming, soccer, and nutrition. The itinerary has been thoughtfully crafted to offer activities that align with each traveler's interests while accommodating weather conditions and maintaining the overall budget of 300, with total planned costs coming to 249.

**Day 1 – March 1 (Partly Cloudy, 22°C):**  
Kick off your trip with two great options: an outdoor Riverfront Soccer Skills Clinic perfect for Kaoru to get involved in soccer drills and teamwork by the river with an indoor backup plan if weather changes. In the evening, relax and enjoy the Evening Comedy Showcase featuring local comedians, which aligns with Yuri’s love of comedy and offers a cozy indoor setting to end the day on a lighthearted note.

**Day 2 – March 2 (Sunny, 24°C):**  
Start the morning with a Sunrise Photography Walk through historic streets and scenic viewpoints, ideal for Hiro’s passion for photography and art. Later, Kaoru can take part in an outdoor AgentsVille Soccer Friendly match offering a lively and social sporting experience.

**Day 3 – March 3 (Rainy, 20°C):**  
With a rainy forecast, indoor activities are a perfect fit. Yuri can sharpen their tennis skills at the Indoor Tennis Skills Lab, while both Yuri and Kaoru can also enjoy a cozy Cooking & Nutrition Class focused on healthy, hearty meals perfect for a rainy day.

**Day 4 – March 4 (Cloudy, 18°C):**  
Hiro’s love for reading and art is well-matched with a Reading Salon & City Stories event in the morning hosted in a literary café, followed by an evening Photography Studio workshop to enhance creative skills with lighting and composition techniques.

**Day 5 – March 5 (Thunderstorm, 19°C):**  
Indoor micro-lectures on math puzzles, logic, and everyday science provide Kaoru with an intellectually stimulating option ideal for stormy weather. Hiro can participate later in the day at a Creative Writing Cafe offering a relaxed, cozy atmosphere to share and explore writing and reading interests.

**Day 6 – March 6 (Partly Cloudy, 21°C):**  
Engage Yuri and Kaoru’s tech and programming interests at the Makerspace Open Lab, an indoor hands-on session with 3D printers and electronics kits. Meanwhile, Hiro can experience a Backstage Theatre Q&A to dive deeper into the world of theatre and art during the evening.

**Day 7 – March 7 (Clear, 23°C):**  
Benefit from the clear weather with an outdoor Sunrise Tennis Rally for Yuri, featuring friendly drills and games. Later, Hiro can enjoy a Live Music Picnic in the afternoon, combining outdoor music performances with a relaxed and social vibe perfect for music lovers.

**Day 8 – March 8 (Rainy, 17°C):**  
On your final day, stay indoors with a Comfort Cooking & Nutrition Lab for Yuri and Kaoru, focusing on nourishing comfort foods suitable for rainy weather. Kaoru can also join an Indoor Soccer Futsal session to finish the trip with an active and social sporting experience.

This itinerary balances indoor and outdoor activities, catering to everyone’s interests while adapting to varying weather conditions and offering a variety of cultural, social, intellectual, and physical experiences. Enjoy your adventure in AgentsVille!

## CONGRATULATIONS! 🎉🥳👏

You have successfully planned a stellar vacation to AgentsVille! Your AI travel agent has demonstrated advanced reasoning techniques, including role-based prompting, chain-of-thought reasoning, ReAct prompting, and feedback loops

Give yourself a pat on the back for completing this project and completing this course!